# RQ1 and RQ2 | Copilot access and pull request merge time

**RQ1. Among merged GitHub pull requests, how do title and body length, change size, programming-language composition, review activity, and merge duration differ by the author's recorded Copilot seat status? Do those differences persist within repositories represented by both groups?**

Put the same cleaned CSV used for RQ2 at `data/cleaned_public.csv` in the repository root, then run the notebook in order. The default works when Jupyter starts in the repository root or in `paper2/`. If the CSV lives elsewhere, set `CSV_PATH` in the first code cell. It reads only the analysis columns in chunks and retains those compact columns for exact median calculations. It excludes raw titles and bodies. It needs Python, pandas, NumPy, and Jupyter. Use the machine that hosts the dataset if your laptop still lacks memory for the reduced table.

Seat status represents recorded access. The comparison is observational and covers merged PRs. Record when seat status was measured if dataset documentation supplies that information. Review results must be filled from the real dataset.

In [ ]:
import json
from ast import literal_eval
from functools import lru_cache
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

CSV_PATH = next(
    (path for path in (Path("data/cleaned_public.csv"), Path("../data/cleaned_public.csv"))
     if path.is_file()),
    Path("data/cleaned_public.csv"),
)
if not CSV_PATH.is_file():
    raise FileNotFoundError(
        "Set CSV_PATH to your cleaned_public.csv or place it in the repository root under data/."
    )
CHUNK_SIZE = 75_000
EXPECTED_PRS = 1_338_401
EXPECTED_REPOSITORIES = 59_474
BOOTSTRAP_REPS = 500
TOP_LANGUAGES = 10
SEED = 42

KEYS = ["repository_id", "pull_request_id"]
SEAT = "had_copilot_seat"
METRICS = [
    "title_word_count", "body_word_count", "files_touched", "lines_modified",
    "code_lines_modified", "config_lines_modified", "num_reviewers",
    "num_review_comments", "minutes_to_merge",
]
FLAGS = ["is_open_source", SEAT, "has_title", "has_body"]
REQUIRED = KEYS + ["user_id", "languages_list"] + METRICS + FLAGS
FLAG_VALUES = {"true": 1, "false": 0, "1": 1, "0": 0,
               "1.0": 1, "0.0": 0, "yes": 1, "no": 0}

def flag(series):
    cleaned = series.astype("string").str.strip().str.lower()
    invalid = cleaned.notna() & ~cleaned.isin(["", *FLAG_VALUES])
    if invalid.any():
        raise ValueError(f"Unexpected values in {series.name}: {cleaned[invalid].unique()[:5]}")
    return cleaned.map(FLAG_VALUES).astype("Int8")

@lru_cache(maxsize=16_384)
def parse_language_list(text):
    try:
        values = json.loads(text)
    except ValueError:
        try:
            values = literal_eval(text)
        except (ValueError, SyntaxError, RecursionError):
            return None
    if not isinstance(values, (list, tuple)):
        return None
    return tuple(dict.fromkeys(str(item).strip() for item in values
                               if item is not None and str(item).strip()))

def load_prs(path):
    sections = []
    rows_read = 0
    reader = pd.read_csv(path, usecols=REQUIRED, chunksize=CHUNK_SIZE,
                         dtype={col: "string" for col in KEYS + ["user_id", "languages_list"]})
    for part in reader:
        rows_read += len(part)
        public = flag(part.pop("is_open_source")).eq(1).fillna(False)
        part = part.loc[public].copy()
        for col in KEYS + ["user_id", "languages_list"]:
            part[col] = part[col].str.strip().replace("", pd.NA)
        for col in FLAGS[1:]:
            part[col] = flag(part[col])
        for col in METRICS:
            number = pd.to_numeric(part[col], errors="coerce").astype("float64")
            part[col] = number.where(np.isfinite(number) & number.ge(0))
        keep = part[KEYS].notna().all(axis=1) & part["minutes_to_merge"].notna()
        sections.append(part.loc[keep, :].copy())
    if not sections:
        raise ValueError("The CSV has no rows.")
    prs = pd.concat(sections, ignore_index=True)
    rows_before_dedup = len(prs)
    prs = prs.drop_duplicates(ignore_index=True)
    if prs.duplicated(KEYS).any():
        raise ValueError("Conflicting retained records share a repository and PR ID.")
    if prs.empty:
        raise ValueError("No public merged PRs with valid identifiers and duration were retained.")
    audit = {
        "source_rows": rows_read,
        "eligible_rows_before_deduplication": rows_before_dedup,
        "duplicate_projected_rows_removed": rows_before_dedup - len(prs),
        "retained_prs": len(prs),
        "retained_repositories": prs.repository_id.nunique(),
        "matches_rq2_reported_counts": (
            len(prs) == EXPECTED_PRS and prs.repository_id.nunique() == EXPECTED_REPOSITORIES
        ),
    }
    return prs, audit

prs, audit = load_prs(CSV_PATH)
display(pd.Series(audit, name="Cohort audit").to_frame())
if not audit["matches_rq2_reported_counts"]:
    print("Check the input version and exclusions before describing RQ1 and RQ2 as sharing a cohort.")

## RQ1 | Group support and pooled distributions

The pooled summary includes PRs with known seat status. Title and body length use recorded word counts. Title and body presence are shown separately. A language list of `[]` means zero listed languages. A blank or unparseable list is missing. Language presence is measured over PRs with a parseable list, and one PR may contribute to multiple languages.

In [ ]:
def indicator(series, predicate):
    return predicate(series).where(series.notna()).astype("float64")

prs["languages"] = prs.pop("languages_list").map(parse_language_list, na_action="ignore")
prs["language_count"] = prs.languages.map(
    lambda values: len(values) if isinstance(values, tuple) else np.nan
)

prs["zero_reviews"] = indicator(prs.num_reviewers, lambda x: x.eq(0))
prs["zero_review_comments"] = indicator(prs.num_review_comments, lambda x: x.eq(0))
prs["zero_minute_merge"] = indicator(prs.minutes_to_merge, lambda x: x.eq(0))
prs["seat_group"] = prs[SEAT].map({0: "No recorded seat", 1: "Recorded seat"}).fillna("Unknown")

def group_support(data):
    return data.groupby("seat_group").agg(
        PRs=("pull_request_id", "size"),
        repositories=("repository_id", "nunique"),
        known_authors=("user_id", "nunique"),
        missing_author_rows=("user_id", lambda x: x.isna().sum()),
        missing_language_rows=("language_count", lambda x: x.isna().sum()),
    )

display(group_support(prs))
known = prs.loc[prs[SEAT].notna()].copy()
if known[SEAT].nunique() != 2:
    raise ValueError("Both recorded seat groups are needed for RQ1 contrasts.")

author_seats = known.groupby("user_id", dropna=True)[SEAT].nunique()
print("Known author IDs appearing in both seat groups:", int(author_seats.eq(2).sum()))

numeric_outcomes = METRICS + ["language_count"]
summaries = []
for outcome in numeric_outcomes:
    grouped = known.groupby("seat_group")[outcome]
    table = grouped.agg(valid="count", median="median")
    for label, quantile in [("p25", .25), ("p75", .75), ("p90", .90)]:
        table[label] = grouped.quantile(quantile)
    table["missing"] = grouped.size() - table["valid"]
    table.insert(0, "outcome", outcome)
    summaries.append(table.reset_index())
pooled_distributions = pd.concat(summaries, ignore_index=True)
display(pooled_distributions.round(2))

## RQ1 | Within-repository comparison

Each effect is recorded seat minus no recorded seat. For each outcome, a repository is eligible if it contains valid observations from both groups. The table shows the pooled difference among all PRs with known status, the pooled difference among PRs in eligible repositories, and the average of differences computed separately inside those same repositories. Within-repository estimates give each eligible repository equal weight. A sensitivity column weights by the number of eligible PRs.

The 95% percentile intervals resample eligible repositories. They reflect repository-level sampling variability and do not account for authors contributing to multiple repositories or multiplicity across outcomes. Use effect sizes and report coverage for each outcome. Language prevalence is shown for the most common programming languages pooled across both groups.

In [ ]:
from collections import Counter

languages = Counter()
for values in known.languages.dropna():
    languages.update(values)
top_languages = sorted(languages, key=lambda language: (-languages[language], language))[:TOP_LANGUAGES]
binary_outcomes = ["has_title", "has_body", "zero_reviews", "zero_review_comments", "zero_minute_merge"]

rng = np.random.default_rng(SEED)

def contrast(data, values, statistic, scale):
    frame = data[["repository_id", SEAT]].assign(value=values).dropna(subset=["value"])
    pooled = frame.groupby(SEAT).value.agg(statistic)
    cells = frame.groupby(["repository_id", SEAT]).value.agg([statistic, "size"])
    paired = cells[statistic].unstack(SEAT).reindex(columns=[0, 1]).dropna()
    eligible = frame.loc[frame.repository_id.isin(paired.index)]
    pooled_eligible = eligible.groupby(SEAT).value.agg(statistic)
    effect = (paired[1] - paired[0]).to_numpy(dtype=float) * scale
    weights = cells["size"].groupby(level="repository_id").sum().reindex(paired.index)
    interval = [np.nan, np.nan]
    if len(effect) >= 2:
        boot = [rng.choice(effect, size=len(effect), replace=True).mean()
                for _ in range(BOOTSTRAP_REPS)]
        interval = np.quantile(boot, [.025, .975])

    def difference(result):
        return float((result.get(1, np.nan) - result.get(0, np.nan)) * scale)

    return {
        "pooled_all": difference(pooled),
        "pooled_eligible": difference(pooled_eligible),
        "within_repo": effect.mean() if effect.size else np.nan,
        "ci_low": interval[0],
        "ci_high": interval[1],
        "within_repo_size_weighted": np.average(effect, weights=weights) if effect.size else np.nan,
        "eligible_repos": len(paired),
        "eligible_no_seat_prs": int(eligible[SEAT].eq(0).sum()),
        "eligible_seat_prs": int(eligible[SEAT].eq(1).sum()),
    }

contrasts = []
for name in numeric_outcomes:
    contrasts.append({"outcome": name, "units": "difference in medians",
                      **contrast(known, known[name], "median", 1)})
for name in binary_outcomes:
    contrasts.append({"outcome": name, "units": "percentage points",
                      **contrast(known, known[name].astype("float64"), "mean", 100)})
for name in top_languages:
    present = known.languages.map(
        lambda values: float(name in values) if isinstance(values, tuple) else np.nan
    )
    contrasts.append({"outcome": f"Language: {name}", "units": "percentage points",
                      **contrast(known, present, "mean", 100)})

rq1_contrasts = pd.DataFrame(contrasts)
display(rq1_contrasts.round(3))

## RQ2 | Completed manuscript results

RQ2 asks how well limited PR metadata classify merge duration in unseen repositories. The manuscript's A100 experiment uses the same 192,854 test PRs for each model below. The overall cohort contains 1,338,401 merged PRs from 59,474 repositories. The training-derived class cutoffs are 35 and 826 minutes.

| Model and feature set | Test accuracy | Test macro F1 |
| --- | ---: | ---: |
| Majority baseline | 41.65% | 0.1960 |
| Logistic regression, change and description metadata | 39.51% | 0.3830 |
| Logistic regression, plus final review counts | 52.97% | 0.4943 |
| XGBoost, change metadata | 43.63% | 0.4067 |
| XGBoost, plus description length and presence | 45.50% | 0.4298 |
| XGBoost, plus final review counts | 54.13% | 0.5076 |

**Interpretation.** Change and description metadata provide a modest signal across unseen repositories. Final reviewer and comment counts improve both models, but XGBoost recalls only 26.0% of medium-duration PRs. Adding recorded seat status changes XGBoost test macro F1 by +0.0024 with description features and −0.0008 when final review counts are included. Feature snapshots at opening are unverified, and final review counts accumulate during the PR lifecycle. The results support retrospective classification among merged PRs.

**Text features.** RQ1 uses word counts and text presence. RQ3 examines what PR titles and bodies describe.